<a href="https://colab.research.google.com/github/Av605/Finance/blob/main/GREEKS_AND_VOLATILITY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ==============================================================================
# 1. ENVIRONMENT SETUP & DEPENDENCY INSTALLATION
# ==============================================================================
!pip install yfinance plotly pandas numpy scipy

import numpy as np
import pandas as pd
import scipy.stats as si
from scipy.optimize import brentq
import yfinance as yf
import datetime
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

warnings.filterwarnings('ignore')
print("Environment successfully configured with required dependencies.")

Environment successfully configured with required dependencies.


In [4]:
# ==============================================================================
# 2. ANALYTICAL PRICING ENGINE (BLACK-SCHOLES & BINOMIAL TREE FROM SCRATCH)
# ==============================================================================

class OptionsEngine:
    """
    A quantitative engine for pricing European and American options,
    computing closed-form Greeks, and deriving implied volatilities.
    """

    @staticmethod
    def black_scholes_price(S, K, T, r, sigma, option_type='call'):
        """Prices a European option using the closed-form Black-Scholes-Merton formula."""
        if T <= 0 or sigma <= 0:
            return max(0.0, S - K) if option_type == 'call' else max(0.0, K - S)

        d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
        d2 = d1 - sigma * np.sqrt(T)

        if option_type.lower() == 'call':
            return S * si.norm.cdf(d1) - K * np.exp(-r * T) * si.norm.cdf(d2)
        else:
            return K * np.exp(-r * T) * si.norm.cdf(-d2) - S * si.norm.cdf(-d1)

    @staticmethod
    def compute_greeks(S, K, T, r, sigma, option_type='call'):
        """Computes analytical closed-form Greeks for a European option."""
        greeks = {'delta': 0.0, 'gamma': 0.0, 'theta': 0.0, 'vega': 0.0, 'rho': 0.0}
        if T <= 0:
            if option_type.lower() == 'call':
                greeks['delta'] = 1.0 if S > K else 0.0
            else:
                greeks['delta'] = -1.0 if S < K else 0.0
            return greeks

        d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
        d2 = d1 - sigma * np.sqrt(T)
        pdf_d1 = si.norm.pdf(d1)

        # Delta
        if option_type.lower() == 'call':
            greeks['delta'] = si.norm.cdf(d1)
        else:
            greeks['delta'] = si.norm.cdf(d1) - 1.0

        # Gamma (Same for Call and Put)
        greeks['gamma'] = pdf_d1 / (S * sigma * np.sqrt(T))

        # Vega (Same for Call and Put) - Scaled for 1% change in vol
        greeks['vega'] = (S * np.sqrt(T) * pdf_d1) / 100.0

        # Theta - Scaled for 1-day passage of time
        if option_type.lower() == 'call':
            theta_calc = -(S * pdf_d1 * sigma) / (2 * np.sqrt(T)) - r * K * np.exp(-r * T) * si.norm.cdf(d2)
        else:
            theta_calc = -(S * pdf_d1 * sigma) / (2 * np.sqrt(T)) + r * K * np.exp(-r * T) * si.norm.cdf(-d2)
        greeks['theta'] = theta_calc / 365.0

        # Rho - Scaled for 1% change in interest rates
        if option_type.lower() == 'call':
            greeks['rho'] = (K * T * np.exp(-r * T) * si.norm.cdf(d2)) / 100.0
        else:
            greeks['rho'] = (-K * T * np.exp(-r * T) * si.norm.cdf(-d2)) / 100.0

        return greeks

    @staticmethod
    def binomial_tree_price(S, K, T, r, sigma, option_type='call', exercise_style='american', N=100):
        """Prices options using the Cox-Ross-Rubinstein (CRR) Binomial Tree model."""
        if T <= 0:
            return max(0.0, S - K) if option_type == 'call' else max(0.0, K - S)

        dt = T / N
        u = np.exp(sigma * np.sqrt(dt))
        d = 1 / u
        p = (np.exp(r * dt) - d) / (u - d)

        # Ensure arbitrage free condition inside the lattice parameters
        if p < 0 or p > 1:
            return OptionsEngine.black_scholes_price(S, K, T, r, sigma, option_type)

        # Initialize asset prices at maturity terminal nodes
        asset_prices = S * (u ** np.arange(N, -1, -1)) * (d ** np.arange(0, N + 1))

        # Initialize option values at maturity
        if option_type.lower() == 'call':
            option_values = np.maximum(0, asset_prices - K)
        else:
            option_values = np.maximum(0, K - asset_prices)

        # Backward induction steps
        for i in range(N - 1, -1, -1):
            asset_prices = S * (u ** np.arange(i, -1, -1)) * (d ** np.arange(0, i + 1))
            continuation_values = np.exp(-r * dt) * (p * option_values[:-1] + (1 - p) * option_values[1:])

            if exercise_style.lower() == 'american':
                if option_type.lower() == 'call':
                    intrinsic_values = np.maximum(0, asset_prices - K)
                else:
                    intrinsic_values = np.maximum(0, K - asset_prices)
                option_values = np.maximum(intrinsic_values, continuation_values)
            else:
                option_values = continuation_values

        return option_values[0]

    @staticmethod
    def calculate_implied_volatility(market_price, S, K, T, r, option_type='call'):
        """Solves for implied volatility using Brent's root-finding method."""
        # Minimum price bounds test
        intrinsic = max(0.0, S - K) if option_type == 'call' else max(0.0, K - S)
        if market_price <= intrinsic:
            return np.nan

        def objective_function(sigma):
            return OptionsEngine.black_scholes_price(S, K, T, r, sigma, option_type) - market_price

        try:
            # Search bounds for reasonable vol structures (0.01% to 500% IV)
            return brentq(objective_function, 1e-4, 5.0)
        except ValueError:
            return np.nan

print("Pricing structures and mathematical functions successfully compiled.")

Pricing structures and mathematical functions successfully compiled.


In [5]:
# ==============================================================================
# 3. LIVE FINANCIAL PIPELINE & DATA HARVESTING
# ==============================================================================

def fetch_option_surface_data(ticker_symbol="SPY"):
    """
    Fetches comprehensive options chain data across all available expiries
    and handles data alignment, mid-price calculation, and filtering.
    """
    print(f"Connecting to data servers for Ticker: {ticker_symbol}...")
    ticker = yf.Ticker(ticker_symbol)

    try:
        # Get historical close for spot representation
        history = ticker.history(period="1d")
        if history.empty:
            raise ValueError("No historical price available for symbol.")
        spot_price = history['Close'].iloc[-1]
    except Exception as e:
        print(f"Error gathering asset base price: {e}. Defaulting SPY proxy value.")
        spot_price = 540.0 # Standard defensive fallback configuration

    # Approximated 13-week T-Bill risk-free rate
    risk_free_rate = 0.045

    expiries = ticker.options
    # Filter to look out max 180 days to avoid ultra-thin deep structures
    max_expiry_days = 180
    today = datetime.date.today()

    all_chains = []
    print(f"Current Underlier Spot: ${spot_price:.2f} | Risk-Free Rate: {risk_free_rate*100:.1f}%")
    print(f"Processing available options maturities...")

    # Iterate across expiries and collect structural frames
    for date_str in expiries[:8]: # Scan closest 8 expiries to maintain responsive execution bounds
        expiry_date = datetime.datetime.strptime(date_str, "%Y-%m-%d").date()
        dte = (expiry_date - today).days

        if dte <= 2 or dte > max_expiry_days:
            continue # Eliminate prompt decay or highly distant tail liquidity issues

        T = dte / 365.0
        chain = ticker.option_chain(date_str)

        for opt_type, df in [('call', chain.calls), ('put', chain.puts)]:
            # Standard cleaning filters for operational stability
            df = df.dropna(subset=['bid', 'ask', 'strike'])
            df['mid'] = (df['bid'] + df['ask']) / 2.0

            # Liquidity constraints: bid must exist, minimum threshold spreads
            df = df[(df['bid'] > 0.05) & (df['volume'] > 5) & (df['mid'] > 0.10)]

            # Focus on liquid moneyness grid near the spot price (±25%)
            df = df[(df['strike'] >= spot_price * 0.75) & (df['strike'] <= spot_price * 1.25)]

            for _, row in df.iterrows():
                K = row['strike']
                market_price = row['mid']

                # Numeric calculation of Implied Volatility
                iv = OptionsEngine.calculate_implied_volatility(market_price, spot_price, K, T, risk_free_rate, opt_type)

                if not np.isnan(iv) and 0.02 < iv < 1.50:
                    greeks = OptionsEngine.compute_greeks(spot_price, K, T, risk_free_rate, iv, opt_type)
                    american_price = OptionsEngine.binomial_tree_price(spot_price, K, T, risk_free_rate, iv, opt_type, 'american')

                    all_chains.append({
                        'DTE': dte,
                        'T': T,
                        'Strike': K,
                        'Type': opt_type,
                        'MarketPrice': market_price,
                        'IV': iv,
                        'AmericanPrice': american_price,
                        'Delta': greeks['delta'],
                        'Gamma': greeks['gamma'],
                        'Theta': greeks['theta'],
                        'Vega': greeks['vega'],
                        'Rho': greeks['rho']
                    })

    processed_df = pd.DataFrame(all_chains)
    print(f"Data ingestion complete. Processed {len(processed_df)} compliant contract variables.")
    return processed_df, spot_price, risk_free_rate

# Run ingestion pipeline
raw_data, spot, r = fetch_option_surface_data("SPY")

Connecting to data servers for Ticker: SPY...
Current Underlier Spot: $743.29 | Risk-Free Rate: 4.5%
Processing available options maturities...
Data ingestion complete. Processed 607 compliant contract variables.


In [6]:
# ==============================================================================
# 4. IMPLIED VOLATILITY SURFACE VISUALIZATION ENGINE
# ==============================================================================

def plot_implied_volatility_surface(df, spot_price):
    """Generates an interactive 3D surface plot mapping IV against Strike and Expiry."""
    # Pivot datasets for Call structural mapping
    calls = df[df['Type'] == 'call'].sort_values(by=['Strike', 'DTE'])

    if calls.empty:
        print("Insufficient structured option data available to render visual surface profiles.")
        return

    # Create uniform grid interpolation structures
    strikes_grid = np.linspace(calls['Strike'].min(), calls['Strike'].max(), 30)
    dte_grid = np.sort(calls['DTE'].unique())
    X, Y = np.meshgrid(strikes_grid, dte_grid)

    # Standard 2D linear interpolation algorithm for grid optimization
    from scipy.interpolate import griddata
    Z = griddata(
        points=(calls['Strike'].values, calls['DTE'].values),
        values=calls['IV'].values,
        xi=(X, Y),
        method='linear'
    )

    # Backfill edge boundaries for visually complete surface rendering
    Z = pd.DataFrame(Z).bfill().ffill().values

    fig = go.Figure(data=[go.Surface(
        x=X, y=Y, z=Z,
        colorscale='Viridis',
        colorbar_title='Implied Vol (IV)',
        hovertemplate='Strike: %{x}<br>DTE: %{y}<br>IV: %{z:.2%}<extra></extra>'
    )])

    fig.update_layout(
        title=f'Institutional 3D Implied Volatility Surface (SPY Proxy Baseline Asset)',
        scene=dict(
            xaxis_title='Strike Price ($)',
            yaxis_title='Days to Expiration (DTE)',
            zaxis_title='Implied Volatility (IV)',
            camera=dict(eye=dict(x=1.5, y=1.5, z=1.2))
        ),
        margin=dict(l=0, r=0, b=0, t=50),
        width=950, height=650
    )
    fig.show()

plot_implied_volatility_surface(raw_data, spot)

# ==============================================================================
# 5. MARKET MICROSTRUCTURE ANALYSIS (SMILE & SKEW CURVES)
# ==============================================================================

def plot_volatility_smile_and_term_structure(df, spot_price):
    """Plots localized cross-sectional slices to isolate Volatility Skew and Term Structure."""
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Volatility Smile/Skew (Fixed Maturity Slice)", "ATM Volatility Term Structure")
    )
    
    # Subplot 1: Select the nearest liquid expiry cycle
    unique_dtes = sorted(df['DTE'].unique())
    target_dte = unique_dtes[1] if len(unique_dtes) > 1 else unique_dtes[0]
    slice_df = df[(df['DTE'] == target_dte) & (df['Type'] == 'call')].sort_values('Strike')
    
    fig.add_trace(
        go.Scatter(x=slice_df['Strike'], y=slice_df['IV'], mode='lines+markers',
                   name=f'{target_dte} DTE Smile', line=dict(color='firebrick', width=3)),
        row=1, col=1
    )
    fig.add_shape(
        type="line", x0=spot_price, y0=slice_df['IV'].min()*0.9, x1=spot_price, y1=slice_df['IV'].max()*1.1,
        line=dict(color="RoyalBlue", width=2, dash="dash"), name="Spot Price", row=1, col=1
    )
    
    # Subplot 2: Term Structure using At-The-Money (ATM) options
    atm_df = df[(df['Type'] == 'call')].copy()
    atm_df['Distance'] = (atm_df['Strike'] - spot_price).abs()
    
    # Isolate minimum moneyness variance per expiration cluster
    term_structure_data = []
    for dte in unique_dtes:
        dte_frame = atm_df[atm_df['DTE'] == dte]
        best_atm = dte_frame.loc[dte_frame['Distance'].idxmin()]
        term_structure_data.append({'DTE': dte, 'IV': best_atm['IV']})
        
    ts_df = pd.DataFrame(term_structure_data)
    
    fig.add_trace(
        go.Scatter(x=ts_df['DTE'], y=ts_df['IV'], mode='lines+markers',
                   name='ATM Term Structure', line=dict(color='darkgreen', width=3)),
        row=1, col=2
    )
    
    fig.update_xaxes(title_text="Strike Price ($)", row=1, col=1)
    fig.update_xaxes(title_text="Days to Expiration (DTE)", row=1, col=2)
    fig.update_yaxes(title_text="Implied Volatility", row=1, col=1)
    fig.update_yaxes(title_text="Implied Volatility", row=1, col=2)
    
    fig.update_layout(title_text="Options Trading Desk Market Skew Dashboards", width=1000, height=450, showlegend=True)
    fig.show()

plot_volatility_smile_and_term_structure(raw_data, spot)

In [7]:
# ==============================================================================
# 6. RISK MANAGEMENT ANALYTICS (THE GREEKS SURFACES)
# ==============================================================================

def plot_greeks_surfaces(df):
    """Constructs dynamic sub-grid visualization structures across system sensitivities."""
    calls = df[df['Type'] == 'call'].sort_values(by=['Strike', 'DTE'])

    strikes_grid = np.linspace(calls['Strike'].min(), calls['Strike'].max(), 30)
    dte_grid = np.sort(calls['DTE'].unique())
    X, Y = np.meshgrid(strikes_grid, dte_grid)

    from scipy.interpolate import griddata

    def interpolate_greek_surface(greek_name):
        Z = griddata(
            points=(calls['Strike'].values, calls['DTE'].values),
            values=calls[greek_name].values,
            xi=(X, Y),
            method='linear'
        )
        return pd.DataFrame(Z).bfill().ffill().values

    delta_Z = interpolate_greek_surface('Delta')
    gamma_Z = interpolate_greek_surface('Gamma')
    vega_Z = interpolate_greek_surface('Vega')

    # Initialize multi-scene rendering matrix
    fig = make_subplots(
        rows=1, cols=3,
        specs=[[{'type': 'surface'}, {'type': 'surface'}, {'type': 'surface'}]],
        subplot_titles=('Delta Surface', 'Gamma Surface', 'Vega Surface')
    )

    fig.add_trace(go.Surface(x=X, y=Y, z=delta_Z, colorscale='Blues', showscale=False), row=1, col=1)
    fig.add_trace(go.Surface(x=X, y=Y, z=gamma_Z, colorscale='Reds', showscale=False), row=1, col=2)
    fig.add_trace(go.Surface(x=X, y=Y, z=vega_Z, colorscale='YlGnBu', showscale=False), row=1, col=3)

    fig.update_layout(
        title='Institutional Risk Dashboard: Portfolio Greek Exposure Surfaces (Calls)',
        width=1100, height=500,
        scene=dict(xaxis_title='Strike', yaxis_title='DTE', zaxis_title='Delta'),
        scene2=dict(xaxis_title='Strike', yaxis_title='DTE', zaxis_title='Gamma'),
        scene3=dict(xaxis_title='Strike', yaxis_title='DTE', zaxis_title='Vega')
    )
    fig.show()

plot_greeks_surfaces(raw_data)

In [8]:
# ==============================================================================
# 7. ADVANCED VOLATILITY SURFACE GRADIENTS & LOCAL VOLATILITY EXTRACTION
# ==============================================================================

def analyze_surface_microstructure(df):
    """
    Computes numerical partial derivatives across the implied volatility surface
    to evaluate skew dynamics (dIV/dK) and term structure decay (dIV/dT).
    """
    calls = df[df['Type'] == 'call'].copy()
    if len(calls) < 50:
        print("Insufficient grid density to compute surface gradients reliably.")
        return

    # Set up a structured mesh grid for clean numeric derivation
    strikes = np.sort(calls['Strike'].unique())
    dtes = np.sort(calls['DTE'].unique()) / 365.0 # Year scale

    # Re-interpolate onto a clean, differentiable grid
    from scipy.interpolate import LinearNDInterpolator
    interp = LinearNDInterpolator(list(zip(calls['Strike'], calls['DTE']/365.0)), calls['IV'])

    # We will pick a middle slice to perform finite differences
    target_strike = strikes[len(strikes) // 2]
    target_dte = dtes[len(dtes) // 2]

    h_k = 0.5   # Strike step
    h_t = 1/365 # Time step (1 day)

    # Compute localized partial derivatives (Finite Differences)
    iv_base = interp(target_strike, target_dte)
    iv_k_up = interp(target_strike + h_k, target_dte)
    iv_k_dn = interp(target_strike - h_k, target_dte)
    iv_t_up = interp(target_strike, target_dte + h_t)

    if np.isnan([iv_base, iv_k_up, iv_k_dn, iv_t_up]).any():
        print("Selected grid midpoint lies outside interpolated bounds. Defaulting to empirical approximation.")
        dIV_dK = (calls['IV'].iloc[-1] - calls['IV'].iloc[0]) / (calls['Strike'].iloc[-1] - calls['Strike'].iloc[0])
        dIV_dT = 0.05
    else:
        dIV_dK = (iv_k_up - iv_k_dn) / (2 * h_k)
        dIV_dT = (iv_t_up - iv_base) / h_t

    print("==============================================================================")
    print("                 VOLATILITY SURFACE SECOND-ORDER METRICS                      ")
    print("==============================================================================")
    print(f"Target Underlier Grid Node  : Strike ${target_strike:.2f} | Time: {target_dte*365:.1f} DTE")
    print(f"Skew Slope (dIV/dK)         : {dIV_dK:.6f} (Negative values validate equity put skew)")
    print(f"Term Structure Slope (dIV/dT): {dIV_dT:.6f} (Quantifies calendar roll yield expectations)")
    print("------------------------------------------------------------------------------")

analyze_surface_microstructure(raw_data)

                 VOLATILITY SURFACE SECOND-ORDER METRICS                      
Target Underlier Grid Node  : Strike $743.00 | Time: 6.0 DTE
Skew Slope (dIV/dK)         : -0.001982 (Negative values validate equity put skew)
Term Structure Slope (dIV/dT): -1.248718 (Quantifies calendar roll yield expectations)
------------------------------------------------------------------------------


In [9]:
# ==============================================================================
# 8. DYNAMIC PORTFOLIO DELTA-GAMMA HEDGING SIMULATOR
# ==============================================================================

def run_hedging_simulation(spot_start, strike, expiry_years, r, sigma, option_type='call', days=30):
    """
    Simulates a dynamic Delta-hedging regime for a short option position,
    comparing a daily rebalancing frequency against unhedged risk trajectories.
    """
    np.random.seed(42) # Reproducible path generation
    dt = 1 / 365.0
    steps = days

    # Simulate an asset path via Geometric Brownian Motion (GBM)
    spot_path = [spot_start]
    for _ in range(steps):
        drift = (r - 0.5 * sigma**2) * dt
        shock = sigma * np.sqrt(dt) * np.random.normal()
        spot_path.append(spot_path[-1] * np.exp(drift + shock))

    portfolio_values_hedged = []
    portfolio_values_unhedged = []
    cash_account = 0.0

    # Initial setup at Day 0: Sell 1 Contract (Short Position)
    init_greeks = OptionsEngine.compute_greeks(spot_start, strike, expiry_years, r, sigma, option_type)
    init_price = OptionsEngine.black_scholes_price(spot_start, strike, expiry_years, r, sigma, option_type)

    # Delta hedge: buy 'delta' shares of underlying asset
    shares_held = init_greeks['delta']
    cash_account += init_price          # Received premium from selling option
    cash_account -= shares_held * spot_start # Paid for long stock hedge

    for step in range(1, steps + 1):
        current_spot = spot_path[step]
        time_left = max(0.0, expiry_years - (step * dt))

        current_option_value = OptionsEngine.black_scholes_price(current_spot, strike, time_left, r, sigma, option_type)
        current_greeks = OptionsEngine.compute_greeks(current_spot, strike, time_left, r, sigma, option_type)

        # Track unhedged short performance trajectory
        unhedged_pnl = init_price - current_option_value
        portfolio_values_unhedged.append(unhedged_pnl)

        # Track dynamically hedged portfolio value
        hedged_pnl = (shares_held * current_spot) + cash_account - current_option_value
        portfolio_values_hedged.append(hedged_pnl)

        # Daily Rebalancing Execution
        shares_needed = current_greeks['delta']
        trade_shares = shares_needed - shares_held
        cash_account -= trade_shares * current_spot
        shares_held = shares_needed

    # Generate Comparison Charts
    fig = go.Figure()
    fig.add_trace(go.Scatter(y=portfolio_values_unhedged, mode='lines', name='Unhedged Short Position', line=dict(color='crimson', dash='dot')))
    fig.add_trace(go.Scatter(y=portfolio_values_hedged, mode='lines', name='Daily Delta-Hedged Portfolio', line=dict(color='royalblue', width=2.5)))

    fig.update_layout(
        title=f"Delta Neutral Hedging Backtest Simulator ({days}-Day Horizon)",
        xaxis_title="Simulation Steps (Trading Days Passed)",
        yaxis_title="Net Position Mark-to-Market PnL ($)",
        template="plotly_dark",
        width=950, height=400
    )
    fig.show()

# Execute model assuming standard baseline parameters
run_hedging_simulation(spot_start=spot, strike=spot*1.02, expiry_years=60/365, r=r, sigma=0.18)

In [10]:
# ==============================================================================
# 9. TRADE IDEA GENERATOR & RELATIVE VALUE ARBITRAGE SCANNER
# ==============================================================================

def scan_for_volatility_anomalies(df, spot_price):
    """
    Identifies mispricing signatures where the pricing model deviates from
    market quotes, scanning for elevated vega-rich structures.
    """
    print("Initializing Quantitative Arbitrage Scanner...")
    df['Price_Diff_Pct'] = ((df['AmericanPrice'] - df['MarketPrice']) / df['MarketPrice']).abs()

    # Filter for options with solid Vega sensitivity to target pure volatility plays
    trade_candidates = df[(df['Vega'] > 0.15) & (df['IV'] > 0.05)].copy()
    trade_candidates = trade_candidates.sort_values(by='Price_Diff_Pct', ascending=False)

    print("\n" + "="*80)
    print("                TOP 3 STRUCTURAL RELATIVE VALUE MISPRICINGS                  ")
    print("="*80)

    if trade_candidates.empty:
        print("No structural pricing anomalies detected meeting baseline filter limits.")
        return

    counter = 0
    for idx, row in trade_candidates.head(3).iterrows():
        counter += 1
        direction = "UNDERVALUED (BUY STRATEGY)" if row['AmericanPrice'] > row['MarketPrice'] else "OVERVALUED (SELL STRATEGY)"

        print(f"ANOMALY SEARCH IDENTIFIER #{counter}")
        print(f"Contract Setup   : SPY {row['DTE']} DTE | Strike: ${row['Strike']} | Type: {row['Type'].upper()}")
        print(f"Market Mid Price : ${row['MarketPrice']:.2f} vs Model Theoretical Target: ${row['AmericanPrice']:.2f}")
        print(f"Implied Vol (IV) : {row['IV']:.1%}")
        print(f"Delta Exposure   : {row['Delta']:.4f} | Vega Risk Factor: {row['Vega']:.4f}")
        print(f"Strategic Action : {direction}")
        print("-"*80)

scan_for_volatility_anomalies(raw_data, spot)

Initializing Quantitative Arbitrage Scanner...

                TOP 3 STRUCTURAL RELATIVE VALUE MISPRICINGS                  
ANOMALY SEARCH IDENTIFIER #1
Contract Setup   : SPY 10 DTE | Strike: $763.0 | Type: CALL
Market Mid Price : $0.34 vs Model Theoretical Target: $0.33
Implied Vol (IV) : 9.9%
Delta Exposure   : 0.0646 | Vega Risk Factor: 0.1552
Strategic Action : OVERVALUED (SELL STRATEGY)
--------------------------------------------------------------------------------
ANOMALY SEARCH IDENTIFIER #2
Contract Setup   : SPY 9 DTE | Strike: $760.0 | Type: CALL
Market Mid Price : $0.50 vs Model Theoretical Target: $0.49
Implied Vol (IV) : 10.1%
Delta Exposure   : 0.0920 | Vega Risk Factor: 0.1927
Strategic Action : OVERVALUED (SELL STRATEGY)
--------------------------------------------------------------------------------
ANOMALY SEARCH IDENTIFIER #3
Contract Setup   : SPY 11 DTE | Strike: $760.0 | Type: PUT
Market Mid Price : $18.06 vs Model Theoretical Target: $18.31
Implied Vol (IV) :

In [11]:
# ==============================================================================
# 10. EXECUTIVE SUMMARY DESK METRICS (PORTFOLIO ANALYTICS OUTPUT)
# ==============================================================================

def generate_desk_executive_summary(df, spot_price):
    """Compiles key portfolio metrics and outputs a clean desk risk report."""
    total_contracts_scanned = len(df)
    call_skew_avg = df[(df['Type'] == 'call') & (df['Strike'] < spot_price)]['IV'].mean()
    put_skew_avg = df[(df['Type'] == 'put') & (df['Strike'] < spot_price)]['IV'].mean()

    mean_iv = df['IV'].mean()
    max_vega_contract = df.loc[df['Vega'].idxmax()]

    print("### RUNNING DESK COMPLIANCE REPORT ###\n")

    summary_data = {
        "Metric Classification": [
            "Baseline Spot Index Value",
            "Total Options Contracts Active in Model",
            "Mean Implied Volatility (System Grid)",
            "Downside Put Implied Vol (Proxy Insurance)",
            "Upside Call Implied Vol (Proxy Yield)",
            "Peak Vega Risk Node (Maturity Destination)"
        ],
        "Value Output": [
            f"${spot_price:.2f}",
            f"{total_contracts_scanned} Contracts",
            f"{mean_iv:.2%}",
            f"{put_skew_avg:.2%}" if not np.isnan(put_skew_avg) else "N/A",
            f"{call_skew_avg:.2%}" if not np.isnan(call_skew_avg) else "N/A",
            f"Strike ${max_vega_contract['Strike']} @ {max_vega_contract['DTE']} DTE"
        ]
    }

    summary_table = pd.DataFrame(summary_data)
    # Renders a clean Markdown table representation directly into the Colab cell console output
    from IPython.display import display, Markdown
    display(Markdown(summary_table.to_markdown(index=False)))

generate_desk_executive_summary(raw_data, spot)

### RUNNING DESK COMPLIANCE REPORT ###



| Metric Classification                      | Value Output           |
|:-------------------------------------------|:-----------------------|
| Baseline Spot Index Value                  | $743.29                |
| Total Options Contracts Active in Model    | 607 Contracts          |
| Mean Implied Volatility (System Grid)      | 17.88%                 |
| Downside Put Implied Vol (Proxy Insurance) | 21.26%                 |
| Upside Call Implied Vol (Proxy Yield)      | 14.37%                 |
| Peak Vega Risk Node (Maturity Destination) | Strike $745.0 @ 11 DTE |

In [12]:
# ==============================================================================
# 11. MODEL CALIBRATION & ROOT-MEAN-SQUARE ERROR (RMSE) DISCOVERY
# ==============================================================================

def execute_model_cross_validation(df):
    """
    Evaluates pricing deviations between Black-Scholes and Binomial Tree models
    to quantify the early-exercise premium of American-style execution.
    """
    valid_data = df.dropna(subset=['MarketPrice', 'AmericanPrice']).copy()

    if valid_data.empty:
        print("Data frame empty. Calibration step skipped.")
        return

    # Calculate absolute differences and European theoretical benchmarks
    bs_prices = []
    for _, row in valid_data.iterrows():
        p = OptionsEngine.black_scholes_price(
            S=spot, K=row['Strike'], T=row['T'], r=r, sigma=row['IV'], option_type=row['Type']
        )
        bs_prices.append(p)

    valid_data['BS_Price'] = bs_prices
    valid_data['Exercise_Premium'] = valid_data['AmericanPrice'] - valid_data['BS_Price']

    # Calculate Root-Mean-Square Error (RMSE) against actual market midpoint quotes
    rmse_bs = np.sqrt(np.mean((valid_data['MarketPrice'] - valid_data['BS_Price'])**2))
    rmse_binom = np.sqrt(np.mean((valid_data['MarketPrice'] - valid_data['AmericanPrice'])**2))

    # Segment data by moneyness to detect structured errors
    valid_data['Moneyness'] = valid_data['Strike'] / spot
    itm_puts = valid_data[(valid_data['Type'] == 'put') & (valid_data['Moneyness'] < 0.95)]

    print("==============================================================================")
    print("                    MODEL METRIC VALIDATION & PERFORMANCE                     ")
    print("==============================================================================")
    print(f"Black-Scholes Framework RMSE vs Market Mid : ${rmse_bs:.4f}")
    print(f"Binomial Tree Lattice Framework RMSE vs Market Mid : ${rmse_binom:.4f}")
    print("------------------------------------------------------------------------------")
    print(f"Average Captured American Early-Exercise Premium  : ${valid_data['Exercise_Premium'].mean():.4f}")
    if not itm_puts.empty:
        print(f"Deep ITM Put Premium Divergence (Exercise Alpha)  : ${itm_puts['Exercise_Premium'].mean():.4f}")
    print("==============================================================================")

    # Plot early exercise premium distributions across strikes
    fig = go.Figure()
    for opt_type, color in [('call', 'rgb(31, 119, 180)'), ('put', 'rgb(214, 39, 40)')]:
        sub_df = valid_data[valid_data['Type'] == opt_type]
        fig.add_trace(go.Scatter(
            x=sub_df['Strike'], y=sub_df['Exercise_Premium'],
            mode='markers', name=f'American Premium ({opt_type.upper()})',
            marker=dict(color=color, size=6, opacity=0.7)
        ))

    fig.update_layout(
        title="Quantifying the American Early-Exercise Premium Grid Across Strikes",
        xaxis_title="Strike Price ($)",
        yaxis_title="Premium Divergence Value ($)",
        template="plotly_dark",
        width=950, height=400
    )
    fig.show()

execute_model_cross_validation(raw_data)

                    MODEL METRIC VALIDATION & PERFORMANCE                     
Black-Scholes Framework RMSE vs Market Mid : $0.0000
Binomial Tree Lattice Framework RMSE vs Market Mid : $0.0551
------------------------------------------------------------------------------
Average Captured American Early-Exercise Premium  : $0.0267
Deep ITM Put Premium Divergence (Exercise Alpha)  : $-0.0023


In [13]:
# ==============================================================================
# 12. HISTORICAL VOLATILITY REALIZATION & PARKINSON PROXY ENGINE
def compute_historical_realized_volatility(ticker_symbol="SPY"):
    """
    Extracts high-density underlying price structures over a rolling 1-year timeline
    to parse annualized realized volatility profiles and Parkinson range metrics.
    """
    print(f"Pulling historical data layers for: {ticker_symbol}...")
    ticker = yf.Ticker(ticker_symbol)
    hist = ticker.history(period="1y")

    if len(hist) < 20:
        print("Historical asset layer insufficient to analyze statistical variance paths.")
        return

    # Standard Close-to-Close Log Return Annualized Volatility Calculations
    hist['Log_Returns'] = np.log(hist['Close'] / hist['Close'].shift(1))
    realized_vol_30d = hist['Log_Returns'].rolling(window=21).std() * np.sqrt(252)
    realized_vol_90d = hist['Log_Returns'].rolling(window=63).std() * np.sqrt(252)

    # Parkinson Volatility Formulation (Uses high-low structural variance for noise insulation)
    parkinson_factor = 1 / (4 * np.log(2))
    hist['Parkinson_Term'] = parkinson_factor * (np.log(hist['High'] / hist['Low']))**2
    parkinson_vol_30d = np.sqrt(hist['Parkinson_Term'].rolling(window=21).mean() * 252)

    current_cc_30 = realized_vol_30d.iloc[-1]
    current_park_30 = parkinson_vol_30d.iloc[-1]

    print("\n" + "="*80)
    print("                    HISTORICAL REALIZED VOLATILITY MATRIX                     ")
    print("="*80)
    print(f"Current Annualized 30-Day Realized Vol (Close-to-Close) : {current_cc_30:.2%}")
    print(f"Current Annualized 90-Day Realized Vol (Close-to-Close) : {realized_vol_90d.iloc[-1]:.2%}")
    print(f"Current Annualized 30-Day Parkinson Range Volatility    : {current_park_30:.2%}")
    print(f"Implied Volatility vs Realized Volatility Spread (SVP)   : {(raw_data['IV'].mean() - current_cc_30):.2%}")
    print("-"*80)
    print("Note: Positive Volatility Spreads point toward systematic variance premium extraction edges.")
    print("="*80)

    # Plot volatility tracking timelines
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=hist.index, y=realized_vol_30d, mode='lines', name='30D Close-to-Close Vol', line=dict(color='orange')))
    fig.add_trace(go.Scatter(x=hist.index, y=parkinson_vol_30d, mode='lines', name='30D Parkinson Range Vol', line=dict(color='cyan', dash='dash')))

    fig.update_layout(
        title=f"Statistical Realized Volatility Trajectory Engine ({ticker_symbol})",
        xaxis_title="Timeline Date Cluster",
        yaxis_title="Annualized Volatility Index Metrics",
        template="plotly_dark",
        width=950, height=400
    )
    fig.show()

compute_historical_realized_volatility("SPY")

Pulling historical data layers for: SPY...

                    HISTORICAL REALIZED VOLATILITY MATRIX                     
Current Annualized 30-Day Realized Vol (Close-to-Close) : 12.73%
Current Annualized 90-Day Realized Vol (Close-to-Close) : 13.14%
Current Annualized 30-Day Parkinson Range Volatility    : 11.09%
Implied Volatility vs Realized Volatility Spread (SVP)   : 5.16%
--------------------------------------------------------------------------------
Note: Positive Volatility Spreads point toward systematic variance premium extraction edges.
